# Phase 6 — optimized Qwen3 QLoRA training

The notebook defaults to a bounded smoke run. Change `RUN_MODE` only after reviewing the preceding run report. Phase 5 held-out examples are used only for a leakage check, never as trainer input.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

RUN_MODE = "smoke"  # smoke -> calibration -> pilot -> full
BRANCH = "refactoring/phase-transformer"
REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_ROOT = Path("/kaggle/temp/text-summarizer/project")
OUTPUT_DIRECTORY = Path("/kaggle/working/text-summarizer/phase6/qwen3-4b-qlora-v1")

print(f"Python: {platform.python_version()}")
print(f"Run mode: {RUN_MODE}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
previous_phase6_configs = sorted(
    path
    for path in Path("/kaggle/input").rglob("phase6-config.json")
    if path.parent.name == "qwen3-4b-qlora-v1"
)
if not OUTPUT_DIRECTORY.exists() and previous_phase6_configs:
    if len(previous_phase6_configs) != 1:
        raise RuntimeError(
            f"Expected at most one previous Phase 6 output; found {previous_phase6_configs}"
        )
    previous_root = previous_phase6_configs[0].parent
    print(f"Restoring previous Phase 6 output from {previous_root}")
    shutil.copytree(previous_root, OUTPUT_DIRECTORY)
elif OUTPUT_DIRECTORY.exists():
    print("Using Phase 6 output already present in /kaggle/working.")
else:
    print("Starting a new Phase 6 workspace.")

In [ ]:
requirements = [
    "transformers>=4.57,<5",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.48,<1",
    "peft>=0.17,<0.19",
    "datasets>=4.3,<5",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("Phase 6 dependencies are ready.")

In [ ]:
if not (PROJECT_ROOT / ".git").is_dir():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
ml_source = str(PROJECT_ROOT / "ml" / "src")
if ml_source not in sys.path:
    sys.path.insert(0, ml_source)
print(f"Project commit: {commit}")

In [ ]:
phase4_manifests = sorted(Path("/kaggle/input").rglob("phase4-manifest.json"))
working_phase4 = Path("/kaggle/working/text-summarizer/phase4/qwen3-sft-v1/phase4-manifest.json")
if working_phase4.is_file():
    phase4_manifests.append(working_phase4)
phase4_manifests = sorted(set(phase4_manifests))
if len(phase4_manifests) != 1:
    raise RuntimeError(f"Expected exactly one Phase 4 input; found {phase4_manifests}")
PHASE4_DIRECTORY = phase4_manifests[0].parent

benchmark_paths = sorted(Path("/kaggle/input").rglob("benchmark-selection.jsonl"))
working_benchmark = Path("/kaggle/working/text-summarizer/phase5/base-benchmark-v1/benchmark-selection.jsonl")
if working_benchmark.is_file():
    benchmark_paths.append(working_benchmark)
benchmark_paths = sorted(set(benchmark_paths))
if len(benchmark_paths) != 1:
    raise RuntimeError(f"Expected exactly one frozen Phase 5 selection; found {benchmark_paths}")
BENCHMARK_SELECTION_PATH = benchmark_paths[0]
print(f"Phase 4 input: {PHASE4_DIRECTORY}")
print(f"Phase 5 leakage check: {BENCHMARK_SELECTION_PATH}")

In [ ]:
from summarizer_ml.fine_tuning import (
    Phase6Config,
    Phase6RunMode,
    prepare_phase6_selection,
)

CONFIG = Phase6Config()
ACTIVE_MODE = Phase6RunMode(RUN_MODE)
SELECTION_DIRECTORY = OUTPUT_DIRECTORY / "selections" / RUN_MODE
selection_result = prepare_phase6_selection(
    PHASE4_DIRECTORY,
    SELECTION_DIRECTORY,
    CONFIG,
    ACTIVE_MODE,
    benchmark_selection_path=BENCHMARK_SELECTION_PATH,
)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
workspace_config_path = OUTPUT_DIRECTORY / "phase6-config.json"
config_path = SELECTION_DIRECTORY / "training-config.json"
config_content = CONFIG.model_dump_json(indent=2) + "\n"
if not workspace_config_path.is_file():
    workspace_config_path.write_text(config_content, encoding="utf-8")
if config_path.is_file() and config_path.read_text(encoding="utf-8") != config_content:
    raise RuntimeError("Existing run-mode selection uses a different configuration.")
config_path.write_text(config_content, encoding="utf-8")
print(selection_result.manifest.model_dump_json(indent=2))

In [ ]:
from datasets import Dataset, load_from_disk
from uuid import uuid4
from summarizer_ml.fine_tuning import tokenize_sft_batch
from summarizer_ml.training_data import SftExample
from transformers import AutoTokenizer

TOKEN_CACHE_DIRECTORY = OUTPUT_DIRECTORY / "token-cache" / RUN_MODE
TRAIN_CACHE = TOKEN_CACHE_DIRECTORY / "train"
VALIDATION_CACHE = TOKEN_CACHE_DIRECTORY / "validation"
cache_input_path = TOKEN_CACHE_DIRECTORY / "cache-input.json"
cache_manifest_path = TOKEN_CACHE_DIRECTORY / "cache-manifest.json"
selection_manifest_path = SELECTION_DIRECTORY / "selection-manifest.json"
selection_sha256 = hashlib.sha256(selection_manifest_path.read_bytes()).hexdigest()
expected_cache_manifest = {
    "model_id": CONFIG.model_id,
    "max_sequence_tokens": CONFIG.max_sequence_tokens,
    "selection_manifest_sha256": selection_sha256,
}

if cache_input_path.is_file():
    actual = json.loads(cache_input_path.read_text(encoding="utf-8"))
    if actual != expected_cache_manifest:
        raise RuntimeError("Existing token cache used different inputs or configuration.")
else:
    if TOKEN_CACHE_DIRECTORY.exists() and any(TOKEN_CACHE_DIRECTORY.iterdir()):
        raise RuntimeError(f"Unidentified token cache: {TOKEN_CACHE_DIRECTORY}")
    TOKEN_CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    cache_input_path.write_text(
        json.dumps(expected_cache_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id)

def load_selection(path):
    return Dataset.from_json(str(path))

def tokenize_batch(batch):
    size = len(batch["record_id"])
    examples = [
        SftExample.model_validate({key: values[index] for key, values in batch.items()})
        for index in range(size)
    ]
    rows = tokenize_sft_batch(examples, tokenizer, CONFIG.max_sequence_tokens)
    return {
        "input_ids": [list(row.input_ids) for row in rows],
        "attention_mask": [list(row.attention_mask) for row in rows],
        "labels": [list(row.labels) for row in rows],
        "length": [row.length for row in rows],
    }

def build_cache(source_path, destination):
    if destination.is_dir():
        return load_from_disk(str(destination))
    raw = load_selection(source_path)
    encoded = raw.map(
        tokenize_batch,
        batched=True,
        batch_size=CONFIG.tokenization_batch_size,
        num_proc=CONFIG.tokenization_processes,
        remove_columns=raw.column_names,
        desc=f"Tokenizing {destination.name}",
    )
    temporary = destination.with_name(f"{destination.name}.building-{uuid4().hex}")
    encoded.save_to_disk(str(temporary))
    temporary.replace(destination)
    return encoded

train_cache = build_cache(selection_result.manifest.train_path, TRAIN_CACHE)
validation_cache = build_cache(selection_result.manifest.validation_path, VALIDATION_CACHE)
cache_manifest_path.write_text(
    json.dumps(expected_cache_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(f"Tokenized train records: {len(train_cache):,}")
print(f"Tokenized validation records: {len(validation_cache):,}")
print(f"Maximum train length: {max(train_cache['length']):,}")

In [ ]:
import torch
from summarizer_ml.fine_tuning import gradient_accumulation_steps, optimizer_steps

if not torch.cuda.is_available():
    raise RuntimeError("Phase 6 requires a Kaggle GPU accelerator.")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT not in {1, 2}:
    raise RuntimeError(f"Expected one or two GPUs; found {GPU_COUNT}")
for index in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {properties.name}; memory={properties.total_memory / (1024**3):.1f} GiB")
accumulation = gradient_accumulation_steps(CONFIG, GPU_COUNT)
planned_steps = (
    CONFIG.smoke_steps if ACTIVE_MODE is Phase6RunMode.SMOKE
    else CONFIG.calibration_steps if ACTIVE_MODE is Phase6RunMode.CALIBRATION
    else optimizer_steps(selection_result.manifest.train_records, CONFIG, GPU_COUNT)
)
print(f"Gradient accumulation: {accumulation}")
print(f"Effective batch: {CONFIG.per_device_train_batch_size * GPU_COUNT * accumulation}")
print(f"Planned optimizer steps: {planned_steps:,}")

In [ ]:
RUN_DIRECTORY = OUTPUT_DIRECTORY / "runs" / RUN_MODE
trainer_script = PROJECT_ROOT / "ml" / "scripts" / "train_qlora.py"
command = [
    sys.executable,
    "-m",
    "accelerate.commands.launch",
    "--num_processes",
    str(GPU_COUNT),
    "--num_machines",
    "1",
    "--mixed_precision",
    "fp16",
]
if GPU_COUNT > 1:
    command.append("--multi_gpu")
command.extend([
    str(trainer_script),
    "--config", str(config_path),
    "--selection-manifest", str(selection_manifest_path),
    "--train-cache", str(TRAIN_CACHE),
    "--validation-cache", str(VALIDATION_CACHE),
    "--output-directory", str(RUN_DIRECTORY),
])
print("Launching:", " ".join(command))
training_environment = os.environ.copy()
training_environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
training_environment["PYTHONPATH"] = os.pathsep.join(
    value for value in [ml_source, training_environment.get("PYTHONPATH")] if value
)
subprocess.run(command, check=True, cwd=PROJECT_ROOT, env=training_environment)

In [ ]:
from summarizer_ml.fine_tuning import optimizer_steps, project_runtime

report_path = RUN_DIRECTORY / "training-report.json"
if not report_path.is_file():
    raise RuntimeError(f"Training report is missing: {report_path}")
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2, sort_keys=True))
if ACTIVE_MODE is Phase6RunMode.CALIBRATION:
    measured_seconds = float(report["train_metrics"]["train_runtime"])
    steps_per_second = float(report["train_metrics"]["train_steps_per_second"])
    measured_steps = max(1, round(measured_seconds * steps_per_second))
    pilot_steps = optimizer_steps(CONFIG.pilot_train_records, CONFIG, GPU_COUNT)
    projection = project_runtime(measured_steps, measured_seconds, pilot_steps)
    print("Pilot runtime projection:")
    print(projection.model_dump_json(indent=2))

In [ ]:
required = [
    RUN_DIRECTORY / "COMPLETED",
    RUN_DIRECTORY / "training-report.json",
    RUN_DIRECTORY / "training-history.jsonl",
    RUN_DIRECTORY / "adapter" / "adapter_config.json",
    RUN_DIRECTORY / "adapter" / "adapter_model.safetensors",
    RUN_DIRECTORY / "tokenizer" / "tokenizer_config.json",
]
for path in required:
    print(path.relative_to(OUTPUT_DIRECTORY), "exists:", path.is_file())
    if not path.is_file():
        raise RuntimeError(f"Missing Phase 6 artifact: {path}")
if not report["losses_finite"]:
    raise RuntimeError("Phase 6 loss gate failed.")
print(f"Phase 6 {RUN_MODE} artifact gate passed.")